## Drawing Abstract Syntax Trees with Graphviz

A discriminated union distinguishes leaves from labelled branches. Node addresses are lists of child indices, so repeated labels still receive distinct identities. The diagram is generated by the native `dot` program installed in the course environment.

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#load "../Support/Visualization.fsx"
open Visualization

### Number nodes in preorder

The tree shape supplies stable addresses. This is the helper’s implementation:

```fsharp
 addresses distinguish repeated labels without relying on object identity.
let assignNumbers tree =
    let rec visit address next tree numbers =
        let numbers = Map.add address next numbers
        match tree with
        | Leaf _ -> next + 1, numbers
        | Node (_, children) ->
            children
            |> List.indexed
            |> List.fold (fun (counter, known) (index, child) ->
                visit (address @ [index]) counter child known) (next + 1, numbers)
    visit [] 0 tree Map.empty |> snd

let treeToDot tree =
    let numbers = assignNumbers tree
    let lines = StringBuilder("digraph AST {\n  node [shape=box];\n")
    let rec visit address tree =
        let id = numbers[address]
        let label, children =
            match tree with
            | Leaf label -> label, []
            | Node (label, children) -> label, children
        lines.AppendLine(sprintf "  n%d [label=%s];" id (quote label)) |> ignore
        for index, child in List.indexed children do
            let childAddress = address @ [index]
            lines.AppendLine(sprintf "  n%d -> n%d;" id numbers[childAddress]) |> ignore
            visit childAddress child
    visit [] tree
    lines.AppendLine("}").ToString()

```

In [ ]:
let tree = Node ("+", [Leaf "x"; Node ("*", [Leaf "x"; Leaf "3"])])
let addresses = assignNumbers tree
checkNotebook (Map.count addresses = 5)
checkNotebook (addresses[[]] = 0 && addresses[[1; 1]] = 4)
let dot = treeToDot tree
checkNotebook (dot.Contains "n0 -> n1" && dot.Contains "n2 -> n4")
printfn "%s" dot

### Render an SVG

`renderDot` starts Graphviz directly, writes graph source through standard input and reads SVG output. Labels are quoted, and arguments never pass through a shell. The kernel displays the returned SVG inline.

In [ ]:
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(renderDot dot), "text/html") |> ignore